In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torchaudio

from tqdm.auto import tqdm
import pandas as pd
import numpy as np
import librosa
import ast
import os
from typing import Dict, List
import subprocess




# from phoneme_encoder import *
# from tajweed_eval import *

In [2]:
# from google.colab import drive
# drive.mount('/content/drive')

In [3]:
# !unzip "/content/drive/MyDrive/Quran_ds.zip" -d /content

In [4]:
MODEL_PATH = "tajweed_ctc_CNN_model.pth"
DATASET_PATH='../../datasets/Quran_ds/audio/audio'
TRAIN_DS_PATH='quran_train_mini.csv'
TEST_DS_PATH='quran_test.csv'


# MODEL_PATH='drive/MyDrive/tajweed_ctc_CNN_model.pth'
# DATASET_PATH='audio/audio'
# TRAIN_DS_PATH='drive/MyDrive/quran_train_mini.csv'
# TEST_DS_PATH='drive/MyDrive/quran_test.csv'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

batch_size = 4
sr = 16000


device


device(type='cuda')

In [5]:
SPACE_TOKEN = "<space>"
WASL = "<wasl>"

SHORT_VOWELS: List[str] = ["a", "i", "u"]
LONG_VOWELS: List[str] = ["aa", "ii", "uu"]
TANWEEN: List[str] = ["an", "in", "un"]

CONSONANTS: List[str] = [
    "ʔ",
    "ʔw",
    "b",
    "t",
    "θ",
    "j",
    "ħ",
    "x",
    "d",
    "ð",
    "r",
    "z",
    "s",
    "ʃ",
    "sˤ",
    "dˤ",
    "tˤ",
    "ðˤ",
    "ʕ",
    "ɣ",
    "f",
    "q",
    "k",
    "l",
    "m",
    "n",
    "h",
    "w",
    "y",
]

CV_SHORT: List[str] = [c + v for c in CONSONANTS for v in SHORT_VOWELS]
CV_LONG: List[str] = [c + v for c in CONSONANTS for v in LONG_VOWELS]
CV_TANWEEN: List[str] = [c + v for c in CONSONANTS for v in TANWEEN]
CONSONANTS: List[str] = [
    "ʔ",
    "ʔw",
    "b",
    "t",
    "θ",
    "j",
    "ħ",
    "x",
    "d",
    "ð",
    "r",
    "z",
    "s",
    "ʃ",
    "sˤ",
    "dˤ",
    "tˤ",
    "ðˤ",
    "ʕ",
    "ɣ",
    "f",
    "q",
    "k",
    "l",
    "m",
    "n",
    "h",
    "w",
    "y",
]
SHORT_VOWELS: List[str] = ["a", "i", "u"]
LONG_VOWELS: List[str] = ["aa", "ii", "uu"]
TANWEEN: List[str] = ["an", "in", "un"]
BLANK_TOKEN = "<blank>"




PHONEMES: List[str] = (
    ["sil", SPACE_TOKEN, WASL] + CONSONANTS + CV_SHORT + CV_LONG + CV_TANWEEN
)
IDX2PHONEME: Dict[int, str] = {i: p for i, p in enumerate(PHONEMES)}

PHONEMES_CTC: List[str] = [BLANK_TOKEN] + PHONEMES
phoneme_to_id: Dict[str, int] = {p: i for i, p in enumerate(PHONEMES_CTC)}
blank_id: int = phoneme_to_id[BLANK_TOKEN]


### تسكين الحرف عند الوقف و همزة الوصل عند بدء الكلام

### فأذلهما الشيطان   (المد عند الوصل)

### add silent char

### add selah to madd eval

### لِأُولِي  أَنَا

# بِأَييِّكُمُ

# بِاللَّهِ   إِنَّمَا يَعْمُرُ مَسَاجِدَ اللَّٰهِ مَنْ آمَنَ بِاللَّهِ

In [6]:
# audio_dir = '../../datasets/Quran_ds/audio/audio'
# text_file = "quran-simple_clean.txt"
# output_csv = "final_dataset.csv"

# data= {}
# quran_data={}

# for folders  , s , l in os.walk(audio_dir):
#     if(l.__len__()==0):
#         continue

#     reciter_name=folders.split('/')[-1]
#     data[reciter_name]=l

# with open(text_file, "r", encoding="utf-8") as f:
#     for line in f:
#         l=line.split('|')
#         if(len(l)!=3):
#             continue

#         surah=line.split('|')[0]
#         ayah= line.split('|')[1]

#         if(len(surah)==1):
#             surah= f'00{surah}'
#         elif(len(surah)==2):
#             surah='0'+surah

#         if(len(ayah)==1):
#             ayah= f'00{ayah}'
#         elif(len(ayah)==2):
#             ayah='0'+ayah

#         key=f'{surah+ayah}'
#         quran_data[key]=line.split('|')[2].replace('\n','')

# final_data=[]

# for reciter in data:
#     files= data[reciter]
#     for file in files:



#         if(quran_data.get(file.split('.')[0])):
#             # print(reciter)
#             # print(file.split('.')[0])
#             # print(quran_data.get(file.split('.')[0]))

#             if(file[3]=='000' or file[3:]=='001'):
#                 continue

#             item={
#                 "surah":file[:3],
#                 "ayah":file[3:6],
#                 'path_of_audio':reciter+'/'+file,
#                 'reciter_name':reciter,
#                 "ayah_text":quran_data.get(file.split('.')[0]),
#             }
#             final_data.append(item)


# # Create DataFrame
# df = pd.DataFrame(final_data)

# # Save CSV
# df.to_csv('quran_ds.csv', index=False, encoding="utf-8")

# print("CSV created:", output_csv)

In [7]:
# df = pd.read_csv("quran_train_mini.csv")

# total_seconds = 0
# failed_files = []

# for path in df["path_of_audio"]:
#     try:
#         info = torchaudio.info(DATASET_PATH+'/'+ path)
#         total_seconds += info.num_frames / info.sample_rate
#     except Exception:

#         failed_files.append(path)

# total_hours = total_seconds / 3600

# print(f"Total duration: {total_hours:.2f} hours")
# print(f"Failed files: {len(failed_files)}")


### Total duration (TRAIN): 146.90 hours

### Total duration  (TEST): 10.91 hours

In [8]:
# import importlib
# import phoneme_encoder

# importlib.reload(phoneme_encoder)
# phoneme_encoder.test_encoder()

In [9]:
# import importlib
# import phoneme_encoder

# importlib.reload(phoneme_encoder)

# ph, meta = phoneme_encoder.text_to_phonemes_with_mapping("بِسْمِ اللَّٰهِ الرَّحْمَـٰنِ الرَّحِيمِ الم",for_debug=False)

# ph

In [10]:
# import generate_quran_phoneme
# import importlib

# importlib.reload(generate_quran_phoneme)

# generate_quran_phoneme.generate_phoneme_dataset()

# generate_quran_phoneme.split_ds()

In [11]:
# from phoneme_encoder import  text_to_phonemes_with_mapping
# df= pd.read_csv('quran_test.csv')
# phonemes=[]
# for text in df['ayah_text']:
#     phonemes.append(text_to_phonemes_with_mapping(text=text,for_debug=False))

# df['phonemes']=phonemes
# df.head()
# df.to_csv('quran_test.csv')

In [12]:
def add_noise(signal, noise_level=0.005):
    noise = np.random.randn(len(signal))
    return signal + noise_level * noise

def random_gain(signal):
    gain = np.random.uniform(0.8, 1.2)
    return signal * gain

def convert_to_wav(audio_path):
    wav_path = audio_path.replace(".mp3", ".wav")
    subprocess.run(["ffmpeg", "-i", audio_path, "-ar", "16000", "-ac", "1", wav_path, "-y"],
                   capture_output=True)
    return wav_path

In [13]:
def load_waveform(audio_path, sr=16000, training=True):
    """Load raw waveform and apply signal-level augmentation only."""

    if audio_path.endswith(".mp3"):
        audio_path = convert_to_wav(audio_path)

    signal, _ = librosa.load(audio_path, sr=sr)

    if training and np.random.rand() < 0.5:
        signal = add_noise(signal)

    if training and np.random.rand() < 0.5:
        signal = random_gain(signal)

    # Normalize waveform amplitude to [-1, 1]
    max_val = np.max(np.abs(signal)) + 1e-8
    signal = signal / max_val

    return torch.tensor(signal, dtype=torch.float32)  # shape: (T,)

In [14]:
class TajweedCTCDataset(Dataset):

    def __init__(self, dataframe, dataset_path, phoneme_to_id, training=True):
        self.df = dataframe
        self.dataset_path = dataset_path
        self.phoneme_to_id = phoneme_to_id
        self.training = training

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        waveform = load_waveform(
            self.dataset_path + "/" + row["path_of_audio"],
            training=self.training,
        )

        phoneme_seq, _ = ast.literal_eval(row["phonemes"])
        target_ids = [
            self.phoneme_to_id[p] for p in phoneme_seq if p in self.phoneme_to_id
        ]

        return (
            waveform,                                       # (T_samples,)
            torch.tensor(target_ids, dtype=torch.long),
            waveform.shape[0],                              # raw sample count
            len(target_ids),
        )

In [15]:
def ctc_collate(batch):
    waveforms, targets, input_lengths, target_lengths = zip(*batch)

    # pad waveforms to same length within batch
    padded_waveforms = pad_sequence(waveforms, batch_first=True)  # (B, T_max)

    return (
        padded_waveforms,
        torch.cat(targets),
        torch.tensor(input_lengths, dtype=torch.long),
        torch.tensor(target_lengths, dtype=torch.long),
    )

In [16]:
class CNNFrontEnd(nn.Module):

    def __init__(self, out_channels=128):
        super().__init__()

        self.cnn = nn.Sequential(
            # Layer 1 — coarse temporal structure
            nn.Conv1d(1, 32, kernel_size=11, stride=5, padding=5),
            nn.BatchNorm1d(32),
            nn.GELU(),

            # Layer 2 — mid-level patterns
            nn.Conv1d(32, 64, kernel_size=7, stride=4, padding=3),
            nn.BatchNorm1d(64),
            nn.GELU(),

            # Layer 3 — fine phonetic features
            nn.Conv1d(64, out_channels, kernel_size=5, stride=4, padding=2),
            nn.BatchNorm1d(out_channels),
            nn.GELU(),

            # Layer 4 — refine without further downsampling
            nn.Conv1d(out_channels, out_channels, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm1d(out_channels),
            nn.GELU(),
        )

    def forward(self, x):
        # x: (B, T_samples)
        x = x.unsqueeze(1)          # → (B, 1, T_samples)
        x = self.cnn(x)             # → (B, out_channels, T')
        x = x.permute(0, 2, 1)     # → (B, T', out_channels)  ← LSTM expects this
        return x

    def output_length(self, input_length):
        L = input_length
        for kernel, stride, padding in [(11,5,5), (7,4,3), (5,4,2), (3,2,1)]:
            L = (L + 2 * padding - kernel) // stride + 1
        return max(1, L)  # ← clamp to avoid 0-length sequences crashing CTC

In [17]:
class SpecAugment(nn.Module):
    """
    Conservative SpecAugment tuned for Tajweed:
    - Small time masks to preserve madd duration information
    - Small freq masks to preserve emphatic/nasal phoneme signatures
    - Multiple masks instead of one large one
    """

    def __init__(
        self,
        time_mask_max=8,      # max 80ms erased — safe for short vowels (~150ms)
        freq_mask_max=10,     # max 10/128 channels — preserves spectral shape
        num_time_masks=2,     # two small time masks instead of one big one
        num_freq_masks=2,     # two small freq masks
    ):
        super().__init__()
        self.time_mask_max = time_mask_max
        self.freq_mask_max = freq_mask_max
        self.num_time_masks = num_time_masks
        self.num_freq_masks = num_freq_masks

    def forward(self, x):
        # x: (B, T, C)
        if not self.training:
            return x

        B, T, C = x.shape
        x = x.clone()

        # Multiple small time masks
        for _ in range(self.num_time_masks):
            t = np.random.randint(1, self.time_mask_max + 1)
            t0 = np.random.randint(0, max(1, T - t))
            x[:, t0:t0 + t, :] = 0

        # Multiple small frequency masks
        for _ in range(self.num_freq_masks):
            f = np.random.randint(1, self.freq_mask_max + 1)
            f0 = np.random.randint(0, max(1, C - f))
            x[:, :, f0:f0 + f] = 0

        return x

In [18]:
class CNNBiLSTM_CTC(nn.Module):

    def __init__(self, cnn_out_channels, hidden_dim, num_layers, num_classes):
        super().__init__()

        self.cnn = CNNFrontEnd(out_channels=cnn_out_channels)
        self.spec_augment = SpecAugment(
                                time_mask_max=8,    # was 30 — now safe for madd
                                freq_mask_max=10,   # was 20 — now safe for emphatics/ghunna
                                num_time_masks=2,
                                num_freq_masks=2,
                            )

        self.lstm = nn.LSTM(
            input_size=cnn_out_channels,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.3,
        )

        self.dropout = nn.Dropout(p=0.3)
        self.classifier = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, waveform, input_lengths):
        # ── CNN front-end ──────────────────────────────────────────────────
        features = self.cnn(waveform)                   # (B, T', cnn_out_channels)

        # ── SpecAugment (only active during model.train()) ─────────────────
        features = self.spec_augment(features)

        # ── Convert raw sample lengths → CNN frame lengths for CTC ─────────
        cnn_lengths = torch.tensor(
            [self.cnn.output_length(int(l)) for l in input_lengths],
            dtype=torch.long,
            device=waveform.device,
        )

        # ── BiLSTM ────────────────────────────────────────────────────────
        packed = nn.utils.rnn.pack_padded_sequence(
            features, cnn_lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        packed_out, _ = self.lstm(packed)
        outputs, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)

        # ── Classifier ────────────────────────────────────────────────────
        outputs = self.dropout(outputs)
        logits = self.classifier(outputs)               # (B, T', num_classes)
        logits = logits.permute(1, 0, 2)                # (T', B, num_classes) for CTC

        return logits, cnn_lengths

In [19]:
train_df = pd.read_csv(TRAIN_DS_PATH)
val_df = pd.read_csv(TEST_DS_PATH)

In [20]:
train_dataset = TajweedCTCDataset(dataframe= train_df,training=True,dataset_path=DATASET_PATH,phoneme_to_id=phoneme_to_id)
val_dataset = TajweedCTCDataset(dataframe= val_df,training=False,dataset_path=DATASET_PATH,phoneme_to_id=phoneme_to_id)

train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, collate_fn=ctc_collate
)

val_loader = DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False, collate_fn=ctc_collate
)

In [21]:
next(enumerate(train_loader))

(0,
 (tensor([[-0.0010, -0.0100, -0.0051,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0175, -0.0072,  0.0058,  ..., -0.0089,  0.0021,  0.0101],
          [ 0.0018, -0.0045, -0.0166,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0009,  0.0075,  0.0019,  ...,  0.0000,  0.0000,  0.0000]]),
  tensor([114, 102,  28, 192,  42, 114,   9,  48, 111,  43,  27, 183,  33, 105,
           12, 117, 108, 183, 102,  87, 156,  63,   6, 127,  33,  29, 117,  30,
           58, 117, 196,  69, 201,  33,  16,  69, 127, 103, 114,  48,  51, 146,
           40, 198, 114,  16,  42,  32,  96, 108,   7, 198,  33,  29,  95,  71,
          113,  28,  86,  27, 279, 114,  89, 104,  31, 288,  93,  29,  86,  14,
           99,  32,  93, 186, 108, 174,  97,  39,  27, 107,  24,  70, 145, 108,
           13, 111,   6,  34, 189,  94,  14,  87,  31, 108,  34,  29, 108, 200,
           81, 177,  93,  51,  72,  63,  93, 195, 144]),
  tensor([272928, 300931, 114103,  50574]),
  tensor([38, 46, 16,  7])))

In [22]:
model = CNNBiLSTM_CTC(
    cnn_out_channels=128,
    hidden_dim=256,
    num_layers=3,
    num_classes=len(phoneme_to_id),
).to(device)

In [23]:
model.state_dict

<bound method Module.state_dict of CNNBiLSTM_CTC(
  (cnn): CNNFrontEnd(
    (cnn): Sequential(
      (0): Conv1d(1, 32, kernel_size=(11,), stride=(5,), padding=(5,))
      (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): GELU(approximate='none')
      (3): Conv1d(32, 64, kernel_size=(7,), stride=(4,), padding=(3,))
      (4): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): GELU(approximate='none')
      (6): Conv1d(64, 128, kernel_size=(5,), stride=(4,), padding=(2,))
      (7): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (8): GELU(approximate='none')
      (9): Conv1d(128, 128, kernel_size=(3,), stride=(2,), padding=(1,))
      (10): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (11): GELU(approximate='none')
    )
  )
  (spec_augment): SpecAugment()
  (lstm): LSTM(128, 256, num_layers=3, batch_first=True, dropout

In [24]:
ctc_loss = nn.CTCLoss(blank=blank_id, zero_infinity=True, reduction="mean")
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

In [ ]:
# checkpoint = torch.load(MODEL_PATH, map_location=torch.device(device))
# model.load_state_dict(checkpoint)

In [26]:
def ctc_decode(frame_preds, blank_id):

    decoded = []
    prev = None

    for p in frame_preds:

        p = int(p)

        if p != blank_id and p != prev:
            decoded.append(p)

        prev = p

    return decoded

In [27]:
def train_model(
    model, train_loader, val_loader, ctc_loss, optimizer, scheduler,
    device, model_path, epochs=20, patience=4,
):
    best_val_loss = float("inf")
    epochs_no_improve = 0

    for epoch in range(epochs):

        # ── Train ──────────────────────────────────────────────────────────
        model.train()
        train_loss = 0

        for waveforms, targets, input_lengths, target_lengths in tqdm(train_loader):
            waveforms      = waveforms.to(device)
            targets        = targets.to(device)
            input_lengths  = input_lengths.to(device)
            target_lengths = target_lengths.to(device)

            optimizer.zero_grad()

            logits, cnn_lengths = model(waveforms, input_lengths)  # ← unpack both
            log_probs = torch.log_softmax(logits, dim=2)

            loss = ctc_loss(log_probs, targets, cnn_lengths, target_lengths)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)

        # ── Validate ───────────────────────────────────────────────────────
        model.eval()
        val_loss = 0

        with torch.no_grad():
            for waveforms, targets, input_lengths, target_lengths in tqdm(val_loader):
                waveforms      = waveforms.to(device)
                targets        = targets.to(device)
                input_lengths  = input_lengths.to(device)
                target_lengths = target_lengths.to(device)

                logits, cnn_lengths = model(waveforms, input_lengths)
                log_probs = torch.log_softmax(logits, dim=2)
                loss = ctc_loss(log_probs, targets, cnn_lengths, target_lengths)
                val_loss += loss.item()

        avg_val_loss = val_loss / len(val_loader)
        scheduler.step(avg_val_loss)
        current_lr = optimizer.param_groups[0]['lr']
        print(f"  LR: {current_lr:.2e}")

        # ── Save / Early stop ──────────────────────────────────────────────
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), model_path)
            print("✅ Model saved")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"⏹ Early stopping at epoch {epoch + 1}")
                break

        print(f"\nEpoch {epoch + 1}/{epochs}")
        print(f"  Train Loss : {avg_train_loss:.4f}")
        print(f"  Val Loss   : {avg_val_loss:.4f}")
        print("-" * 40)

In [28]:
train_model(
    model, train_loader, val_loader,
    ctc_loss, optimizer, scheduler,
    device, MODEL_PATH, epochs=30, patience=4,
)

  0%|          | 0/2501 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# def phoneme_alignment(ref, hyp):

#     n = len(ref)
#     m = len(hyp)

#     dp = [[0] * (m + 1) for _ in range(n + 1)]

#     for i in range(n + 1):
#         dp[i][0] = i
#     for j in range(m + 1):
#         dp[0][j] = j

#     for i in range(1, n + 1):
#         for j in range(1, m + 1):

#             if ref[i - 1] == hyp[j - 1]:
#                 cost = 0
#             else:
#                 cost = 1

#             dp[i][j] = min(
#                 dp[i - 1][j] + 1,  # deletion
#                 dp[i][j - 1] + 1,  # insertion
#                 dp[i - 1][j - 1] + cost,  # substitution
#             )

#     return dp

In [ ]:
# def extract_errors(ref, hyp):

#     dp = phoneme_alignment(ref, hyp)

#     i = len(ref)
#     j = len(hyp)

#     errors = []

#     while i > 0 or j > 0:

#         if i > 0 and j > 0 and ref[i - 1] == hyp[j - 1]:
#             i -= 1
#             j -= 1

#         elif i > 0 and dp[i][j] == dp[i - 1][j] + 1:
#             errors.append(
#                 {"type": "Deletion", "phoneme": ref[i - 1], "position": i - 1}
#             )
#             i -= 1

#         elif j > 0 and dp[i][j] == dp[i][j - 1] + 1:
#             errors.append(
#                 {"type": "Insertion", "phoneme": hyp[j - 1], "position": j - 1}
#             )
#             j -= 1

#         else:
#             errors.append(
#                 {
#                     "type": "Substitution",
#                     "expected": ref[i - 1],
#                     "predicted": hyp[j - 1],
#                     "position": i - 1,
#                 }
#             )
#             i -= 1
#             j -= 1

#     return errors[::-1]

In [ ]:
# def greedy_decode_with_frames(logits, blank_id):
#     """
#     logits: (T, batch, num_classes)
#     Returns:
#         frame_preds: list of predicted phoneme ids per frame
#         collapsed: phoneme sequence after CTC collapse
#     """

#     pred = torch.argmax(logits, dim=2)  # (T, batch)
#     pred = pred[:, 0].cpu().numpy()  # assume batch size 1 for now

#     frame_preds = pred.tolist()

#     collapsed = []
#     prev = None

#     for p in frame_preds:
#         if p != blank_id and p != prev:
#             collapsed.append(p)
#         prev = p

#     return frame_preds, collapsed

In [ ]:
# def frames_to_segments(frame_preds, blank_id):

#     segments = []
#     current_ph = None
#     count = 0

#     for p in frame_preds:

#         if p == blank_id:
#             if current_ph is not None:
#                 segments.append((current_ph, count))
#                 current_ph = None
#                 count = 0
#             continue

#         if p != current_ph:
#             if current_ph is not None:
#                 segments.append((current_ph, count))
#             current_ph = p
#             count = 1
#         else:
#             count += 1

#     if current_ph is not None:
#         segments.append((current_ph, count))

#     return segments

In [ ]:
# def normalize_segment_durations(segments):
#     """
#     segments: list of (phoneme_id, frame_count)
#     Returns list of (phoneme_id, frame_count, normalized_duration)
#     """

#     total_frames = sum(frames for _, frames in segments)
#     num_segments = len(segments)

#     if num_segments == 0:
#         return []

#     avg_duration = total_frames / num_segments

#     normalized = []

#     for phoneme_id, frames in segments:
#         norm = frames / avg_duration
#         normalized.append((phoneme_id, frames, norm))

#     return normalized

In [ ]:
# def align_phonemes(expected, predicted):
#     """
#     Align expected phoneme sequence with predicted sequence.

#     Returns list of tuples:
#     (expected_phoneme, predicted_phoneme)
#     """

#     n = len(expected)
#     m = len(predicted)

#     # DP table
#     dp = [[0] * (m + 1) for _ in range(n + 1)]

#     for i in range(n + 1):
#         dp[i][0] = i

#     for j in range(m + 1):
#         dp[0][j] = j

#     for i in range(1, n + 1):
#         for j in range(1, m + 1):

#             if expected[i - 1] == predicted[j - 1]:
#                 cost = 0
#             else:
#                 cost = 1

#             dp[i][j] = min(
#                 dp[i - 1][j] + 1,  # deletion
#                 dp[i][j - 1] + 1,  # insertion
#                 dp[i - 1][j - 1] + cost,  # substitution
#             )

#     # Backtrack
#     alignment = []
#     i = n
#     j = m

#     while i > 0 and j > 0:

#         if expected[i - 1] == predicted[j - 1]:
#             alignment.append((expected[i - 1], predicted[j - 1]))
#             i -= 1
#             j -= 1

#         elif dp[i][j] == dp[i - 1][j - 1] + 1:
#             alignment.append((expected[i - 1], predicted[j - 1]))
#             i -= 1
#             j -= 1

#         elif dp[i][j] == dp[i - 1][j] + 1:
#             alignment.append((expected[i - 1], None))
#             i -= 1

#         else:
#             alignment.append((None, predicted[j - 1]))
#             j -= 1

#     alignment.reverse()

#     return alignment

In [ ]:
# def evaluate_madd_rules(rules, segments, tolerance=0.8):

#     avg_short = compute_avg_short_duration(segments)

#     results = []
#     errors = []

#     if avg_short is None:
#         return results, errors

#     for rule in rules:

#         phoneme_index = rule["phoneme_index"]
#         expected = rule["expected_harakat"]

#         if phoneme_index >= len(segments):
#             continue

#         pid, frames = segments[phoneme_index]

#         observed = frames / avg_short

#         if abs(observed - expected) <= tolerance:
#             status = "Correct"
#         elif observed < expected:
#             status = "Too Short"
#         else:
#             status = "Too Long"

#         result = {
#             "phoneme_index": phoneme_index,
#             "type": rule["type"],
#             "expected": expected,
#             "observed": round(observed, 2),
#             "status": status,
#         }

#         results.append(result)

#         if status != "Correct":
#             errors.append(result)

#     return results, errors

In [ ]:
# def word_errors(expected_words, predicted_words):

#     n = len(expected_words)
#     m = len(predicted_words)

#     dp = [[0] * (m + 1) for _ in range(n + 1)]

#     for i in range(n + 1):
#         dp[i][0] = i
#     for j in range(m + 1):
#         dp[0][j] = j

#     for i in range(1, n + 1):
#         for j in range(1, m + 1):

#             if expected_words[i - 1] == predicted_words[j - 1]:
#                 cost = 0
#             else:
#                 cost = 1

#             dp[i][j] = min(dp[i - 1][j] + 1, dp[i][j - 1] + 1, dp[i - 1][j - 1] + cost)

#     # Backtrace
#     i, j = n, m
#     errors = []

#     while i > 0 or j > 0:

#         if i > 0 and j > 0 and expected_words[i - 1] == predicted_words[j - 1]:
#             i -= 1
#             j -= 1

#         elif i > 0 and dp[i][j] == dp[i - 1][j] + 1:
#             errors.append(
#                 {
#                     "type": "Missing Word",
#                     "word": expected_words[i - 1],
#                     "position": i - 1,
#                 }
#             )
#             i -= 1

#         elif j > 0 and dp[i][j] == dp[i][j - 1] + 1:
#             errors.append(
#                 {
#                     "type": "Extra Word",
#                     "word": predicted_words[j - 1],
#                     "position": j - 1,
#                 }
#             )
#             j -= 1

#         else:
#             errors.append(
#                 {
#                     "type": "Wrong Word",
#                     "expected": expected_words[i - 1],
#                     "predicted": predicted_words[j - 1],
#                     "position": i - 1,
#                 }
#             )
#             i -= 1
#             j -= 1

#     return errors[::-1]

In [ ]:
# def evaluate_audio(model, audio_path, ayah_text):

#     model.eval()

#     # ---------------------------------------
#     # 1) Extract audio features
#     # ---------------------------------------

#     features = extract_features(audio_path)

#     features = torch.from_numpy(features).float()

#     input_len = torch.tensor([features.shape[0]], dtype=torch.long)

#     features = features.unsqueeze(0).to(device)

#     # ---------------------------------------
#     # 2) Run ASR model
#     # ---------------------------------------

#     with torch.no_grad():

#         logits = model(features, input_len)  # (T,B,V)
#         frame_preds = torch.argmax(logits, dim=-1)  # (T,B)

#     frame_preds = frame_preds.squeeze(1).cpu().numpy()  # (T,)

#     # ---------------------------------------
#     # 3) Convert frames → phoneme segments
#     # ---------------------------------------

#     segments = frames_to_segments(frame_preds, blank_id)

#     # segments example:
#     # [(pid, frames), (pid, frames), ...]

#     # ---------------------------------------
#     # 4) Predicted phoneme sequence
#     # ---------------------------------------

#     predicted_phonemes = [IDX2PHONEME[pid] for pid, _ in segments]

#     # ---------------------------------------
#     # 5) CTC decode for text comparison
#     # ---------------------------------------

#     decoded_ids = ctc_decode(frame_preds, blank_id)

#     decoded_phonemes = [IDX2PHONEME[i] for i in decoded_ids]

#     predicted_text = phonemes_to_text(decoded_phonemes)

#     # ---------------------------------------
#     # 6) Extract Tajweed rules from ayah text
#     # ---------------------------------------

#     phoneme_seq, phoneme_meta, madd_rules = extract_madd_rules(ayah_text)

#     # phoneme_seq = expected phoneme sequence

#     # ---------------------------------------
#     # 7) Align expected vs predicted phonemes
#     # ---------------------------------------

#     alignment = align_phonemes(phoneme_seq, predicted_phonemes)

#     # alignment example:
#     # {0:0, 1:None, 2:1, 3:2}

#     # ---------------------------------------
#     # 8) Evaluate Madd rules
#     # ---------------------------------------

#     madd_results = []
#     madd_errors = []

#     avg_short = compute_avg_short_duration(segments)

#     if avg_short is not None:

#         for rule in madd_rules:

#             expected_idx = rule["phoneme_index"]
#             expected_harakat = rule["expected_harakat"]

#             pred_idx = alignment.get(expected_idx)

#             if pred_idx is None:
#                 continue

#             if pred_idx >= len(segments):
#                 continue

#             pid, frames = segments[pred_idx]

#             observed = frames / avg_short

#             if abs(observed - expected_harakat) <= 0.8:
#                 status = "Correct"
#             elif observed < expected_harakat:
#                 status = "Too Short"
#             else:
#                 status = "Too Long"

#             result = {
#                 "phoneme_index": expected_idx,
#                 "type": rule["type"],
#                 "expected": expected_harakat,
#                 "observed": round(observed, 2),
#                 "status": status,
#             }

#             madd_results.append(result)

#             if status != "Correct":
#                 madd_errors.append(result)

#     # ---------------------------------------
#     # 9) Word error analysis
#     # ---------------------------------------

#     word_error_list = word_errors(ayah_text, predicted_text)

#     # ---------------------------------------
#     # 10) Debug info (very useful)
#     # ---------------------------------------

#     debug = {
#         "expected_phonemes": phoneme_seq,
#         "predicted_phonemes": predicted_phonemes,
#         "segments": segments,
#         "alignment": alignment,
#     }

#     # ---------------------------------------
#     # 11) Final evaluation result
#     # ---------------------------------------

#     result = {
#         "expected_text": ayah_text,
#         "predicted_text": predicted_text,
#         "word_errors": word_error_list,
#         "madd_results": madd_results,
#         "madd_errors": madd_errors,
#         "debug": debug,
#     }

#     return result

In [ ]:
# print(df.iloc[2]["ayah_text"])

In [ ]:
# evaluate_audio(
#     model=model,
#     audio_path=df.iloc[1]["path_of_audio"],
#     ayah_text=df.iloc[1]["ayah_text"],
# )